# VietHandOCR Part 4: VietOCR Model Training

Welcome to **Part 4** of the VietHandOCR pipeline.
- **Previous Notebook**: [Part 3: Digital Image Processing (DIP) Pipeline](./03_Digital_Image_Processing.ipynb)
- **Next Notebook**: [Part 5: Evaluation & Inference](./05_Evaluation_and_Inference.ipynb)

## Introduction
Here we instantiate the `vgg_transformer` architecture using the VietOCR library. We load pre-trained weights to save massive amounts of compute time. The training loop integrates **Weights & Biases (W&B)** and collects **Out-of-Fold (OOF)** predictions.



In [ ]:
# HOTFIX: Prevent Kaggle Pillow memory corruption error during Save & Run All
import PIL._util
import os
if not hasattr(PIL._util, 'is_directory'):
    PIL._util.is_directory = os.path.isdir

import os, gc, torch
import pandas as pd
import torch.nn as nn
import wandb

# Install VietOCR if not present
# !git clone https://github.com/pbcquoc/vietocr.git
# !pip install -q -e ./vietocr

from vietocr.tool.config import Cfg
from vietocr.model.trainer import Trainer

def setup_vietocr_model():
    '''Loads the vgg_transformer config and builds the model.'''
    config = Cfg.load_config_from_name('vgg_transformer')
    config['device'] = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    
    # Customizing config for our dataset (outputs from Notebook 03)
    config['dataset']['train'] = 'processed_train.txt'
    config['dataset']['valid'] = 'processed_val.txt'
    config['dataset']['image_dir'] = '.'  # Paths in txt are already relative to current dir
    
    print("Model config loaded. Pre-trained weights prepared.")
    return config



In [ ]:
# Training Loop Skeleton with MLOps (W&B, OOF)
wandb.init(project="VietHandOCR", name="vgg-transformer-run1")
config = setup_vietocr_model()

# We use the built-in Trainer to build dataloaders and load pretrained weights properly
trainer = Trainer(config, pretrained=True)
trainer.config['trainer']['epochs'] = 20

# Use the official VietOCR trainer loop
trainer.train()

# After training, load best weights for OOF predictions
model = trainer.model
model.eval()

from vietocr.tool.predictor import Predictor
from PIL import Image

# Setup predictor for OOF / Holdout Validation predictions
pred_config = Cfg.load_config_from_name('vgg_transformer')
pred_config['device'] = config['device']
predictor = Predictor(pred_config)
predictor.model = model  # Swap to newly trained model

print("Collecting Holdout/OOF Predictions...")
valid_file = config['dataset']['valid']
if os.path.exists(valid_file):
    with open(valid_file, 'r', encoding='utf-8') as f:
        val_lines = f.readlines()
    
    oof_predictions = []
    for line in val_lines:
        parts = line.strip().split('\t')
        if len(parts) == 2:
            img_path, label = parts
            if os.path.exists(img_path):
                img = Image.open(img_path)
                pred = predictor.predict(img)
                oof_predictions.append((img_path, label, pred))
    
    import pandas as pd
    pd.DataFrame(oof_predictions).to_csv('holdout_preds_vgg_transformer.txt', sep='\t', index=False, header=False)

wandb.finish()

